# Phase 1: ViewState + Axis Selectors (Rust Backend)

This notebook validates selector updates, normalization, and state hash/version transitions against the Rust daemon.

In [1]:
from __future__ import annotations

import os
import shutil
import sys
import tempfile
from pathlib import Path


def _find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'src').exists() and (candidate / 'tests').exists():
            return candidate
    raise RuntimeError('could not locate repository root from notebook cwd')


REPO_ROOT = _find_repo_root(Path.cwd().resolve())
if str(REPO_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / 'src'))
if str(REPO_ROOT / 'tests') not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / 'tests'))

from lucida.client import LucidaClient
from conftest import create_sample_omezarr
from rust_daemon import start_rust_daemon


In [2]:
tmp_dir = Path(tempfile.mkdtemp(prefix='lucida-phase1-viewstate-rust-'))
dataset_uri = create_sample_omezarr(str(tmp_dir / 'viewstate.zarr'))
daemon = start_rust_daemon(repo_root=REPO_ROOT, env=dict(os.environ))
client = LucidaClient(base_url=daemon.base_url)
print({'dataset_uri': dataset_uri, 'base_url': daemon.base_url})


{'dataset_uri': '/var/folders/hs/qw7ws1q52153c4c639t_p3600000gn/T/lucida-phase1-viewstate-rust-l498nrkl/viewstate.zarr', 'base_url': 'http://127.0.0.1:50540'}


In [3]:
session = client.create_session()
opened = client.open_dataset(dataset_uri, session_id=session.session_id)
created = client.create_view(dataset_id=opened.dataset_summary.dataset_id, session_id=session.session_id)

view_id = created.view_state.view_id
initial = created.view_state
assert initial.state_version == 0
assert bool(initial.state_hash)

{'session_id': session.session_id, 'dataset_id': opened.dataset_summary.dataset_id, 'view_id': view_id}


{'session_id': 'session_cf54cfd609264761',
 'dataset_id': 'ds_7bd2b613e9c4c988',
 'view_id': 'view_3396292ff8274183'}

In [4]:
set_dim = client.set_dim(view_id=view_id, axis='z', index=3, session_id=session.session_id)
set_range = client.set_axis_range(
    view_id=view_id,
    axis='z',
    start=1,
    end_exclusive=4,
    session_id=session.session_id,
)
set_set = client.set_axis_set(view_id=view_id, axis='z', indices=[0, 2, 2], session_id=session.session_id)

assert set_dim.view_state.state_version == 1
assert set_range.view_state.state_version == 2
assert set_set.view_state.state_version == 3
assert set_dim.view_state.state_hash != initial.state_hash
assert set_range.view_state.state_hash != set_dim.view_state.state_hash
assert set_set.view_state.state_hash != set_range.view_state.state_hash

selectors = {item.axis: item for item in set_set.selectors_applied}
assert selectors['z'].kind == 'set'
assert selectors['z'].indices == [0, 2]

{
    'state_version': set_set.view_state.state_version,
    'state_hash': set_set.view_state.state_hash,
    'z_selector': selectors['z'].model_dump(mode='json'),
}


{'state_version': 3,
 'state_hash': '8fe609c585737925426e0ac21e421bc132097edcc7684e226173204643a9c3de',
 'z_selector': {'axis': 'z',
  'kind': 'set',
  'index': None,
  'start': None,
  'end_exclusive': None,
  'indices': [0, 2],
  'clamp': True}}

In [5]:
fetched = client.get_view(view_id=view_id, session_id=session.session_id)
assert fetched.view_state.state_version == set_set.view_state.state_version
assert fetched.view_state.state_hash == set_set.view_state.state_hash
fetched.view_state.model_dump(mode='json')['selectors']


[{'axis': 't',
  'kind': 'index',
  'index': 0,
  'start': None,
  'end_exclusive': None,
  'indices': None,
  'clamp': True},
 {'axis': 'c',
  'kind': 'index',
  'index': 0,
  'start': None,
  'end_exclusive': None,
  'indices': None,
  'clamp': True},
 {'axis': 'z',
  'kind': 'set',
  'index': None,
  'start': None,
  'end_exclusive': None,
  'indices': [0, 2],
  'clamp': True}]

In [6]:
if 'client' in globals():
    client.close()
if 'daemon' in globals():
    daemon.stop()
if 'tmp_dir' in globals():
    shutil.rmtree(tmp_dir, ignore_errors=True)
